# Experiment Claims

**Source files:** `iid_experiment.pkl`, `p03_experiment.pkl`, `p07_experiment.pkl`

---

## Claim 1: The general transformer learns the Laplace estimator.

A transformer trained on sequences drawn uniformly from the full (0,1) interval converges to the same predictions as Laplace. Laplace is Bayes-optimal under a uniform prior, so the match is the result, not a failure. The transformer has internalized an optimal Bayesian rule from data alone. It does not exceed Laplace because representing the full distribution gives it fewer effective examples per source region, which is exactly what the theory predicts.

*Key numbers:* At p=0.3, l=100: transformer 0.9165 BPS, Laplace 0.9144. Pattern holds across all p and all l.

---

## Claim 2: A specialized transformer beats Laplace at small l, but the gap closes.

Training on a single p gives the transformer a prior-like head start. At l=10, p=0.3: transformer 0.9140 vs. Laplace 0.9744. As l grows, the empirical count dominates and Laplace catches up. It is Bayes-optimal and the transformer has no guarantee of updating its implicit estimate as efficiently as a Bayesian posterior update. The transformer's fixed context window also caps what it can condition on at long l.

*Key numbers:* p=0.3 gap at l=10 is 0.060 BPS. Gap at l=499 is 0.007 BPS.


In [53]:
import sys
sys.path.append('../../')
import numpy as np
import torch
import torch.nn.functional as F
from mingpt.model import GPT
import pickle


In [54]:
experiment_save_folder = './experiments/'
N_values = [10, 25, 50, 100, 200, 300, 499]

In [55]:

def iid_generate(p_dict, N):
    p0 = p_dict[0]
    p1 = p_dict[1]
    return np.random.choice([0,1], size=N, p=[p0, p1])

def generate_bin_pmf():
    p = np.random.rand()
    dict_out = {
        0: p,
        1: 1-p
    }
    return dict_out

def convert_sample_to_pmf(sample):
    values, counts = np.unique(sample, return_counts=True)
    probs = counts/counts.sum()
    return dict(zip(values, probs))

def entropy_calc(pmf):
    pmf = clean_pmf(pmf)
    pmf_array = np.asarray(list(pmf.values()))
    return -np.sum(pmf_array * np.log2(pmf_array))

def clean_pmf(pmf):
    return {k: v for k, v in pmf.items() if v > 0.0}

In [56]:
def sequential_universal_source_coding(seq, p_array = None):
    total_bits = 0
    k = 0
    i = 0

    for bit in seq:
        if p_array is not None:
            p1 = p_array[i]
        else:
            p1 = (k+1) / (i+2)
        p0 = 1-p1

        if bit == 1:
            total_bits += -np.log2(p1)
        else:
            total_bits += -np.log2(p0)
        k += bit
        i += 1

    bits_per_symbol = total_bits / len(seq)
    return bits_per_symbol

In [57]:

# generate an array of probabilities given a bit sequence

def transformer_p_array(sequence, model, device='mps'):
    model.eval()
    sequence=torch.tensor(sequence, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        logits, _ = model(sequence)  # [1, T, vocab_size]
        # convert to probabilities
        probs = F.softmax(logits, dim=-1)  # [1, T, vocab_size]
        # extract probability of token=1 at each timestep
        p_array = [0.5] + probs[0, :-1, 1].tolist()
    
    return p_array

In [58]:
def load_model(model_dir):
    checkpoint = torch.load(model_dir, map_location='cpu')
    block_size = checkpoint['block_size']
    
    model_config = GPT.get_default_config()
    model_config.model_type = 'gpt-nano'
    model_config.vocab_size = 2
    model_config.block_size = block_size
    
    model = GPT(model_config)
    model.load_state_dict(checkpoint['state_dict'])
    model = model.to('mps')
    model.eval()
    return model

In [59]:
def markov_generate(p_stay, N):
    state = np.random.choice([0, 1])  # stationary is uniform for symmetric chain
    seq = [state]
    for _ in range(N - 1):
        if np.random.rand() < p_stay:
            seq.append(state)  # stay
        else:
            state = 1 - state  # switch
            seq.append(state)
    return np.array(seq)


In [60]:
# Transformer vs Laplace prior quality at small N

def run_experiment(model, p_values, N_values, output_file, n_trials=100, device='mps'):
    """
    For each (p, N) pair, run n_trials with identical sequences for both methods.
    Returns nested dict: results[p][N] = {'laplace': (mean, std), 'transformer': (mean, std)}
    """
    all_results = {}
    save_dir = experiment_save_folder + output_file
    print(f'Saving to {save_dir}')

    for p_true in p_values:
        print(f"\nRunning p={p_true}")
        p_dict = {0: 1 - p_true, 1: p_true}
        all_results[p_true] = {}

        for N in N_values:
            laplace_bps_trials = []
            transformer_bps_trials = []

            for trial in range(n_trials):
                # Same sequence for both methods
                seq = iid_generate(p_dict, N)

                # Laplace
                bps_lap = sequential_universal_source_coding(seq)
                laplace_bps_trials.append(bps_lap)

                # Transformer
                p_array = transformer_p_array(seq, model, device=device)
                bps_trans = sequential_universal_source_coding(seq, p_array=p_array)
                transformer_bps_trials.append(bps_trans)

            all_results[p_true][N] = {
                'laplace': (np.mean(laplace_bps_trials), np.std(laplace_bps_trials)),
                'transformer': (np.mean(transformer_bps_trials), np.std(transformer_bps_trials)),
                'entropy': entropy_calc({p_true: p_true, 1-p_true: 1-p_true})
            }
            print(f"  N={N}: laplace={np.mean(laplace_bps_trials):.4f}, transformer={np.mean(transformer_bps_trials):.4f}")

    with open(save_dir, 'wb') as f:
        pickle.dump(all_results, f)

    print(f'Saved experiment to {save_dir}')




In [61]:

model = load_model('./models/iid_model.pt')

compress_seq = iid_generate(generate_bin_pmf(), 99)  # 99 because model block_size = n-1 = 99
generated_p = convert_sample_to_pmf(compress_seq)

p_array = transformer_p_array(compress_seq.tolist(), model)
p_array = [0.5] + p_array[:-1]  # assume first bit is uniform (no prior)
bps_transformer = sequential_universal_source_coding(compress_seq, p_array=p_array)

bps_laplace = sequential_universal_source_coding(compress_seq)
bps_entropy = entropy_calc(generated_p)

print(f'Entropy (lower bound): {bps_entropy:.4f}')
print(f'Laplace BPS:           {bps_laplace:.4f}')
print(f'Transformer BPS:       {bps_transformer:.4f}')

number of parameters: 0.11M
Entropy (lower bound): 0.9993
Laplace BPS:           1.0296
Transformer BPS:       1.0266


In [62]:
model_iid = load_model('./models/iid_model.pt')

p_values_iid = [0.1, 0.3, 0.5, 0.7, 0.9]

run_experiment(model_iid, p_values_iid, N_values, output_file='iid_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ./experiments/iid_experiment.pkl

Running p=0.1
  N=10: laplace=0.6279, transformer=0.6289
  N=25: laplace=0.5565, transformer=0.5606
  N=50: laplace=0.4946, transformer=0.4975
  N=100: laplace=0.4944, transformer=0.4964
  N=200: laplace=0.4882, transformer=0.4891
  N=300: laplace=0.4696, transformer=0.4702
  N=499: laplace=0.4791, transformer=0.4797

Running p=0.3
  N=10: laplace=0.9427, transformer=0.9531
  N=25: laplace=0.9355, transformer=0.9406
  N=50: laplace=0.9214, transformer=0.9260
  N=100: laplace=0.9122, transformer=0.9140
  N=200: laplace=0.8985, transformer=0.8995
  N=300: laplace=0.8937, transformer=0.8945
  N=499: laplace=0.8877, transformer=0.8882

Running p=0.5
  N=10: laplace=1.0777, transformer=1.0875
  N=25: laplace=1.0542, transformer=1.0583
  N=50: laplace=1.0369, transformer=1.0390
  N=100: laplace=1.0220, transformer=1.0237
  N=200: laplace=1.0145, transformer=1.0153
  N=300: laplace=1.0103, transformer=1.0108
  N=499: lapl

In [63]:
model_p03 = load_model('./models/p03_model.pt')

p_values_03 = [0.3]

run_experiment(model_p03, p_values_03, N_values, output_file='p03_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ./experiments/p03_experiment.pkl

Running p=0.3
  N=10: laplace=0.9761, transformer=0.9103
  N=25: laplace=0.9333, transformer=0.8806
  N=50: laplace=0.9223, transformer=0.8859
  N=100: laplace=0.9139, transformer=0.8902
  N=200: laplace=0.9004, transformer=0.8860
  N=300: laplace=0.8904, transformer=0.8801
  N=499: laplace=0.8879, transformer=0.8811
Saved experiment to ./experiments/p03_experiment.pkl


In [64]:

model_p07 = load_model('./models/p07_model.pt')

p_values_07 = [0.7]

run_experiment(model_p07, p_values_07, N_values, output_file='p07_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ./experiments/p07_experiment.pkl

Running p=0.7
  N=10: laplace=0.9609, transformer=0.8810
  N=25: laplace=0.9594, transformer=0.9046
  N=50: laplace=0.9135, transformer=0.8734
  N=100: laplace=0.9013, transformer=0.8791
  N=200: laplace=0.8957, transformer=0.8815
  N=300: laplace=0.8960, transformer=0.8860
  N=499: laplace=0.8908, transformer=0.8837
Saved experiment to ./experiments/p07_experiment.pkl


In [65]:
model_points = load_model('./models/points_model.pt')

p_values_points = [0.1, 0.3, 0.5, 0.7, 0.9]

run_experiment(model_points, p_values_points, N_values, output_file='points_experiment.pkl', n_trials=100, device='mps')

number of parameters: 0.11M
Saving to ./experiments/points_experiment.pkl

Running p=0.1
  N=10: laplace=0.6368, transformer=0.6227
  N=25: laplace=0.5626, transformer=0.5487
  N=50: laplace=0.5195, transformer=0.5063
  N=100: laplace=0.4974, transformer=0.4873
  N=200: laplace=0.4968, transformer=0.4906
  N=300: laplace=0.4854, transformer=0.4809
  N=499: laplace=0.4865, transformer=0.4837

Running p=0.3
  N=10: laplace=0.9597, transformer=0.9590
  N=25: laplace=0.9392, transformer=0.9443
  N=50: laplace=0.9124, transformer=0.9167
  N=100: laplace=0.8992, transformer=0.8993
  N=200: laplace=0.8995, transformer=0.8986
  N=300: laplace=0.8884, transformer=0.8874
  N=499: laplace=0.8913, transformer=0.8908

Running p=0.5
  N=10: laplace=1.0924, transformer=1.1038
  N=25: laplace=1.0602, transformer=1.0683
  N=50: laplace=1.0334, transformer=1.0393
  N=100: laplace=1.0231, transformer=1.0271
  N=200: laplace=1.0139, transformer=1.0153
  N=300: laplace=1.0104, transformer=1.0103
  N=499: l